<a href="https://colab.research.google.com/github/Santhosh200307/resume/blob/main/santhoshp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Retry corrected demo code with explicit int() casts where OpenCV expects Python ints.
import cv2
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os
from dataclasses import dataclass, field
from typing import Tuple, List, Any, Optional

class FeatureExtractor:
    def __init__(self, hist_bins=8):
        self.hist_bins = hist_bins

    def extract(self, img_bgr: np.ndarray) -> np.ndarray:
        img = cv2.resize(img_bgr, (128, 128))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)
        feats = [
            float(np.mean(h)), float(np.mean(s)), float(np.mean(v)),
            float(np.std(h)), float(np.std(s)), float(np.std(v))
        ]
        hist = cv2.calcHist([h], [0], None, [self.hist_bins], [0, 180]).flatten()
        hist_norm = (hist / (hist.sum() + 1e-6)).tolist()
        feats.extend([float(x) for x in hist_norm])
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        lap_var = float(lap.var())
        feats.append(lap_var)
        return np.array(feats, dtype=float)

class FreshnessScorer:
    def __init__(self):
        self.w_color = 0.6
        self.w_texture = 0.4

    def score(self, feature_vector: np.ndarray) -> float:
        S_mean = feature_vector[1]
        V_mean = feature_vector[2]
        lap_var = feature_vector[-1]
        s_norm = np.clip(S_mean / 255.0, 0.0, 1.0)
        v_norm = np.clip(V_mean / 255.0, 0.0, 1.0)
        lap_norm = np.arctan(lap_var / 100.0) * (2.0 / np.pi)
        color_freshness = 0.5 * s_norm + 0.5 * v_norm
        score = self.w_color * color_freshness + self.w_texture * lap_norm
        score = float(np.clip(score, 0.0, 1.0))
        return score

class FruitClassifier:
    def __init__(self, n_neighbors=3):
        self.fe = FeatureExtractor(hist_bins=8)
        self.clf = KNeighborsClassifier(n_neighbors=n_neighbors)

    def fit(self, images: List[np.ndarray], labels: List[str]):
        X = [self.fe.extract(img) for img in images]
        self.clf.fit(X, labels)
        return self

    def predict(self, img: np.ndarray) -> Tuple[str, np.ndarray]:
        fv = self.fe.extract(img).reshape(1, -1)
        label = self.clf.predict(fv)[0]
        probs = None
        if hasattr(self.clf, "predict_proba"):
            probs = self.clf.predict_proba(fv)[0]
        return label, fv.flatten()

    def save(self, path: str):
        joblib.dump((self.clf, self.fe), path)

    def load(self, path: str):
        self.clf, self.fe = joblib.load(path)
        return self

class CameraHandler:
    @staticmethod
    def create_solid_circle_image(color_bgr: Tuple[int,int,int], radius=50, size=(200,200)):
        img = np.zeros((size[1], size[0], 3), dtype=np.uint8) + 255
        center = (size[0]//2, size[1]//2)
        cv2.circle(img, center, radius, (int(color_bgr[0]), int(color_bgr[1]), int(color_bgr[2])), -1, cv2.LINE_AA)
        cv2.circle(img, (center[0]-int(radius*0.3), center[1]-int(radius*0.3)), int(radius*0.2), (255,255,255), -1)
        return img

    def demo_source(self):
        samples = []
        labels = []
        fruits = {
            "apple": (30, 30, 220),
            "banana": (30, 220, 220),
            "orange": (10, 140, 255),
            "avacardo":(20,170,235)
        }
        rng = np.random.default_rng(1234)
        for name, bgr in fruits.items():
            for i in range(30):
                color_noise = tuple(max(0, min(255, int(c + int(rng.integers(-20,20))))) for c in bgr)
                radius = int(40 + int(rng.integers(-6, 7)))
                img = self.create_solid_circle_image(color_noise, radius=radius, size=(200,200))
                if rng.random() < 0.25:
                    for _ in range(int(rng.integers(1,4))):
                        x = int(rng.integers(40, 160))
                        y = int(rng.integers(40, 160))
                        col = (int(rng.integers(10,40)), int(rng.integers(10,40)), int(rng.integers(10,40)))
                        cv2.circle(img, (x,y), int(rng.integers(3,10)), col, -1)
                samples.append(img)
                labels.append(name + "_fresh")

            for i in range(10):
                dark = tuple(max(0, int(c * float(rng.uniform(0.4, 0.7)))) for c in bgr)
                radius = int(38 + int(rng.integers(-6, 7)))
                img = self.create_solid_circle_image(dark, radius=radius, size=(200,200))
                for _ in range(int(rng.integers(2,6))):
                    x = int(rng.integers(40, 160))
                    y = int(rng.integers(40, 160))
                    axes = (int(rng.integers(4,12)), int(rng.integers(4,12)))
                    angle = int(rng.integers(0,180))
                    color = (int(rng.integers(10,80)), int(rng.integers(40,100)), int(rng.integers(40,100)))
                    cv2.ellipse(img, (x,y), axes, angle, 0, 360, color, -1)
                samples.append(img)
                labels.append(name + "_rotten")
        return samples, labels

def demo_pipeline():
    cam = CameraHandler()
    samples, labels = cam.demo_source()
    X_imgs = samples
    y = labels
    clf = FruitClassifier(n_neighbors=5)
    clf.fit(X_imgs, y)
    idxs = np.arange(len(X_imgs))
    train_idx, test_idx = train_test_split(idxs, test_size=0.2, random_state=42, stratify=y)
    X_test = [X_imgs[i] for i in test_idx]
    y_test = [y[i] for i in test_idx]
    y_pred = []
    fvs = []
    for img in X_test:
        label, fv = clf.predict(img)
        y_pred.append(label)
        fvs.append(fv)
    print("Accuracy on synthetic test set:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))
    scorer = FreshnessScorer()
    print("\nExample predictions (label, predicted_label, freshness_score):")
    for i in range(10):
        real_label = y_test[i]
        pred_label = y_pred[i]
        fv = fvs[i]
        score = scorer.score(fv)
        real_type = real_label.split("_")[0]
        pred_type = pred_label.split("_")[0]
        print(f"Real: {real_label:15} Pred: {pred_label:15} | Fruit: {real_type:7} PredFruit: {pred_type:7} | Freshness: {score:.3f}")
    model_path = "fruit_knn_demo.joblib"
    clf.save(model_path)
    print(f"\nTrained model saved to: {model_path}")
    return clf, scorer, (X_test, y_test, y_pred)

clf, scorer, test_data = demo_pipeline()


Accuracy on synthetic test set: 0.78125
                 precision    recall  f1-score   support

    apple_fresh       0.86      1.00      0.92         6
   apple_rotten       0.50      0.50      0.50         2
 avacardo_fresh       1.00      0.83      0.91         6
avacardo_rotten       0.50      0.50      0.50         2
   banana_fresh       1.00      0.83      0.91         6
  banana_rotten       0.50      0.50      0.50         2
   orange_fresh       0.83      0.83      0.83         6
  orange_rotten       0.33      0.50      0.40         2

       accuracy                           0.78        32
      macro avg       0.69      0.69      0.68        32
   weighted avg       0.81      0.78      0.79        32


Example predictions (label, predicted_label, freshness_score):
Real: orange_rotten   Pred: banana_rotten   | Fruit: orange  PredFruit: banana  | Freshness: 0.672
Real: banana_fresh    Pred: banana_fresh    | Fruit: banana  PredFruit: banana  | Freshness: 0.422
Real: avaca